# Chapter 5.9: SGLang Frontend DSL — Programming Language for LLMs

## Learning Objectives

By the end of this notebook, you will:

1. **Understand the SGLang DSL philosophy**: LLM programs, not just prompts
2. **Master the `@function` decorator** and program definition
3. **Use `gen()`, `select()`, `fork()/join()`** primitives
4. **Build complex LLM programs** with multi-step reasoning
5. **Understand batched execution** of SGLang programs
6. **Trace the frontend source code** architecture

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.9_frontend_dsl.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.9_frontend_dsl.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. The SGLang DSL Philosophy

Traditional LLM usage:
```python
# Just a prompt -> response
response = model.generate("What is AI?")
```

SGLang's approach:
```python
# A PROGRAM that orchestrates multiple LLM calls
@sgl.function
def research_paper(s, topic):
    s += sgl.system("You are a research assistant.")
    s += sgl.user(f"Write an outline for a paper on {topic}")
    s += sgl.assistant(sgl.gen("outline", max_tokens=300))
    s += sgl.user("Now write the abstract.")
    s += sgl.assistant(sgl.gen("abstract", max_tokens=200))
    s += sgl.user("Rate the quality 1-10.")
    s += sgl.assistant(sgl.gen("rating", regex=r"[0-9]|10"))
```

### Key Ideas

1. **Programs, not prompts**: Multi-step interactions as first-class programs
2. **Named outputs**: Each `gen()` produces a named variable
3. **Constraints**: Use regex, JSON schema on individual generations
4. **Parallelism**: Fork/join for parallel generation
5. **Automatic batching**: Multiple program instances run as a batch
6. **Prefix sharing**: The runtime automatically shares KV-cache for common prefixes

## 2. Core Primitives

### 2.1 `@sgl.function` — Defines an LLM Program

```python
@sgl.function
def my_program(s, arg1, arg2):
    # s is the SGLang state (conversation/context)
    # arg1, arg2 are user-provided arguments
    s += "some text"  # Append text to context
    s += sgl.gen("output_name")  # Generate and name the output
```

### 2.2 `gen()` — Generate Text

```python
sgl.gen(
    name="output_name",      # Name for this output
    max_tokens=100,           # Max tokens to generate
    temperature=0.7,          # Sampling temperature
    top_p=0.95,               # Nucleus sampling
    regex=r"[0-9]+",         # Regex constraint
    stop="\n",               # Stop sequence
)
```

### 2.3 `select()` — Choose from Options

```python
sgl.select(
    name="choice",
    choices=["positive", "negative", "neutral"],
)
# Selects the option with highest probability
```

### 2.4 `fork()/join()` — Parallel Generation

```python
forks = s.fork(3)  # Create 3 parallel branches
for f in forks:
    f += sgl.gen("response", max_tokens=100)
s += sgl.join(forks)  # Wait for all to complete
```

### 2.5 Role Markers

```python
sgl.system("System prompt")     # System message
sgl.user("User message")        # User message
sgl.assistant("Assistant reply") # Assistant message
sgl.image(url_or_data)           # Image input (multimodal)
```

In [ ]:
# Simulate the SGLang DSL (without actual LLM backend)

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable

class SglGenToken:
    """Represents a gen() call."""
    def __init__(self, name, max_tokens=100, regex=None, stop=None, temperature=1.0):
        self.name = name
        self.max_tokens = max_tokens
        self.regex = regex
        self.stop = stop
        self.temperature = temperature
    def __repr__(self):
        constraints = []
        if self.regex: constraints.append(f"regex={self.regex!r}")
        if self.stop: constraints.append(f"stop={self.stop!r}")
        c = ", ".join(constraints)
        return f"gen({self.name!r}, max_tokens={self.max_tokens}{', ' + c if c else ''})"

class SglSelectToken:
    """Represents a select() call."""
    def __init__(self, name, choices):
        self.name = name
        self.choices = choices
    def __repr__(self):
        return f"select({self.name!r}, choices={self.choices})"

class SglRole:
    """Represents a role marker."""
    def __init__(self, role, content):
        self.role = role
        self.content = content
    def __repr__(self):
        return f"{self.role}({self.content!r})"

class SglState:
    """Simulates SGLang program state."""
    
    def __init__(self):
        self.messages = []     # Conversation history
        self.outputs = {}      # Named outputs
        self.trace = []        # Execution trace
    
    def __iadd__(self, other):
        if isinstance(other, str):
            self.messages.append({"type": "text", "content": other})
            self.trace.append(f"append: {other[:50]}..." if len(other) > 50 else f"append: {other}")
        elif isinstance(other, SglRole):
            self.messages.append({"type": "role", "role": other.role, "content": other.content})
            self.trace.append(f"{other.role}: {other.content[:50]}")
        elif isinstance(other, SglGenToken):
            # Simulate generation
            simulated = f"[Generated: {other.name}]"
            self.outputs[other.name] = simulated
            self.messages.append({"type": "gen", "name": other.name})
            self.trace.append(f"gen({other.name}) -> {simulated}")
        elif isinstance(other, SglSelectToken):
            # Simulate selection
            selected = other.choices[0]  # Pick first for demo
            self.outputs[other.name] = selected
            self.trace.append(f"select({other.name}) -> {selected}")
        return self
    
    def __getitem__(self, name):
        return self.outputs.get(name, "")
    
    def display_trace(self):
        print("\nExecution Trace:")
        for i, step in enumerate(self.trace):
            print(f"  {i+1}. {step}")
        print(f"\nOutputs: {self.outputs}")

# Helper functions (simulating sgl module)
def gen(name, max_tokens=100, regex=None, stop=None, temperature=1.0):
    return SglGenToken(name, max_tokens, regex, stop, temperature)

def select(name, choices):
    return SglSelectToken(name, choices)

def system(content):
    return SglRole("system", content)

def user(content):
    return SglRole("user", content)

def assistant(content):
    if isinstance(content, (SglGenToken, SglSelectToken)):
        return content  # Will be handled by __iadd__
    return SglRole("assistant", content)


# Demo: A multi-step LLM program
def analyze_sentiment(s, text):
    """An SGLang program for sentiment analysis."""
    s += system("You are a sentiment analysis expert.")
    s += user(f"Analyze the sentiment of: '{text}'")
    s += assistant(gen("analysis", max_tokens=100))
    s += user("What is the overall sentiment?")
    s += select("sentiment", choices=["positive", "negative", "neutral"])
    s += user("Rate confidence 1-10.")
    s += gen("confidence", regex=r"[0-9]|10")

# Run
state = SglState()
analyze_sentiment(state, "I absolutely love this product!")

print("SGLang Program Execution Demo")
print("=" * 50)
state.display_trace()

print(f"\nAccessing outputs:")
print(f"  state['analysis'] = {state['analysis']}")
print(f"  state['sentiment'] = {state['sentiment']}")
print(f"  state['confidence'] = {state['confidence']}")

## 3. Advanced Patterns

### 3.1 Tree of Thought

```python
@sgl.function
def tree_of_thought(s, question):
    s += sgl.system("Think step by step.")
    s += sgl.user(question)
    
    # Generate 3 different reasoning paths
    forks = s.fork(3)
    for i, f in enumerate(forks):
        f += sgl.assistant(sgl.gen(f"thought_{i}", max_tokens=200))
    
    # Evaluate each path
    s += sgl.join(forks)
    s += sgl.user("Which reasoning path is best? Summarize the answer.")
    s += sgl.assistant(sgl.gen("final_answer", max_tokens=100))
```

### 3.2 Multi-Turn with Context

```python
@sgl.function
def multi_turn_qa(s, document, questions):
    s += sgl.system("Answer questions about the document.")
    s += sgl.user(f"Document: {document}")
    s += sgl.assistant("I've read the document. Ask me anything.")
    
    for i, q in enumerate(questions):
        s += sgl.user(q)
        s += sgl.assistant(sgl.gen(f"answer_{i}", max_tokens=150))
```

### 3.3 Structured Extraction

```python
@sgl.function
def extract_entities(s, text):
    s += sgl.user(f"Extract entities from: {text}")
    s += sgl.assistant(sgl.gen(
        "entities",
        regex=r'\{"persons": \[.*\], "places": \[.*\], "dates": \[.*\]\}'
    ))
```

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatches# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(14, 7))ax.set_xlim(0, 14)ax.set_ylim(0, 9)ax.axis('off')fig.patch.set_facecolor('white')ax.text(7, 8.5, 'SGLang Program Execution: Fork/Join Pattern', fontsize=14,        fontweight='bold', ha='center', color=DARK_GRAY)# Main program startbox = mpatches.FancyBboxPatch((5, 7), 4, 0.8, boxstyle="round,pad=0.1",                               facecolor=BLUE, alpha=0.85, edgecolor='white', lw=2)ax.add_patch(box)ax.text(7, 7.4, 'Main Program\ns += system("...") + user("...")', ha='center', va='center',        fontsize=9, color='white', fontweight='bold')# Forkax.text(7, 6.3, 'FORK(3)', ha='center', fontsize=11, fontweight='bold', color=ORANGE,        bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_ORANGE, alpha=0.5))# Three branchesbranches = [    (2.5, 4.5, 'Branch 1\ngen("response_1")', GREEN),    (7, 4.5, 'Branch 2\ngen("response_2")', ORANGE),    (11.5, 4.5, 'Branch 3\ngen("response_3")', PURPLE),]for x, y, label, color in branches:    box = mpatches.FancyBboxPatch((x-1.8, y-0.5), 3.6, 1, boxstyle="round,pad=0.1",                                   facecolor=color, alpha=0.85, edgecolor='white', lw=2)    ax.add_patch(box)    ax.text(x, y, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')# Arrows from fork to branchesfor x, _, _, _ in branches:    ax.annotate('', xy=(x, 5), xytext=(7, 5.9),                arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=1.5))# Joinax.text(7, 3.3, 'JOIN', ha='center', fontsize=11, fontweight='bold', color=RED,        bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_RED, alpha=0.5))for x, _, _, _ in branches:    ax.annotate('', xy=(7, 3.6), xytext=(x, 4),                arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=1.5))# Select bestbox = mpatches.FancyBboxPatch((4.5, 1.5), 5, 0.8, boxstyle="round,pad=0.1",                               facecolor=RED, alpha=0.85, edgecolor='white', lw=2)ax.add_patch(box)ax.text(7, 1.9, 'Select Best Response\ns += sgl.select(["response_1", ...])', ha='center', va='center',        fontsize=9, color='white', fontweight='bold')ax.annotate('', xy=(7, 2.3), xytext=(7, 3),            arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=1.5))# KV cache sharing noteax.text(7, 0.6, 'All 3 branches share the system prompt KV-cache via RadixAttention',        ha='center', fontsize=10, color=BLUE, fontweight='bold',        bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_BLUE, alpha=0.3))plt.tight_layout()plt.savefig("sglang_fork_join.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Demonstrate advanced patterns

def chain_of_thought(s, question):
    """Chain-of-thought reasoning pattern."""
    s += system("You are an expert problem solver. Think step by step.")
    s += user(question)
    
    # Step 1: Identify the problem type
    s += user("What type of problem is this?")
    s += select("problem_type", choices=["math", "logic", "science", "general"])
    
    # Step 2: Think through the problem
    s += user("Now work through it step by step.")
    s += gen("reasoning", max_tokens=300)
    
    # Step 3: Extract the final answer
    s += user("What is the final answer? Be concise.")
    s += gen("answer", max_tokens=50, stop="\n")

def self_consistency(s, question, num_samples=3):
    """Self-consistency: generate multiple answers and vote."""
    s += system("You are a helpful assistant.")
    s += user(question)
    
    # Generate multiple independent answers
    for i in range(num_samples):
        s += user(f"Attempt {i+1}: Think independently and answer.")
        s += gen(f"attempt_{i}", max_tokens=100, temperature=0.8)
    
    # Vote on the best answer
    s += user("Based on all attempts, what is the consensus answer?")
    s += gen("consensus", max_tokens=50)

# Run chain-of-thought
print("Chain of Thought Pattern")
print("=" * 50)
state = SglState()
chain_of_thought(state, "If a train travels at 60 mph for 2.5 hours, how far does it go?")
state.display_trace()

print("\n\nSelf-Consistency Pattern")
print("=" * 50)
state2 = SglState()
self_consistency(state2, "What is 17 * 23?", num_samples=3)
state2.display_trace()

## 4. Batched Execution

SGLang programs can be executed in batch for higher throughput:

```python
# Single execution
state = my_program.run(arg1="hello")

# Batch execution (all instances share prefix)
states = my_program.run_batch([
    {"arg1": "hello"},
    {"arg1": "world"},
    {"arg1": "test"},
])
```

### How Batching Works with RadixAttention

```
Batch of 3 instances of the same program:

Instance 1: [system_prompt | user("hello")  | gen(analysis) | ...]
Instance 2: [system_prompt | user("world")  | gen(analysis) | ...]
Instance 3: [system_prompt | user("test")   | gen(analysis) | ...]
             └─── shared ──┘ └─ different ─┘

RadixTree:
[root]
  └── [system_prompt]  (KV-cache computed ONCE, shared 3x)
       ├── [user("hello")]  -> gen(analysis)
       ├── [user("world")]  -> gen(analysis)
       └── [user("test")]   -> gen(analysis)

Result: System prompt KV-cache reused for all 3 instances!
```

In [ ]:
# Simulate batch execution with prefix sharing

class BatchRunner:
    """Simulates SGLang batch execution."""
    
    def __init__(self):
        self.prefix_cache = {}  # Simulate RadixCache
    
    def run_batch(self, program_fn, inputs: List[dict]):
        """Run a program for multiple inputs."""
        states = []
        prefix_hits = 0
        total_tokens = 0
        
        print(f"Batch execution: {len(inputs)} instances")
        print("=" * 50)
        
        for i, inp in enumerate(inputs):
            state = SglState()
            program_fn(state, **inp)
            states.append(state)
            
            # Check prefix sharing
            messages_key = str(state.messages[:2])  # First 2 messages (system + user)
            if messages_key in self.prefix_cache:
                prefix_hits += 1
                print(f"  Instance {i}: Prefix cache HIT")
            else:
                self.prefix_cache[messages_key] = True
                print(f"  Instance {i}: Prefix cache MISS (first time)")
        
        print(f"\n  Prefix cache hits: {prefix_hits}/{len(inputs)}")
        print(f"  Cache hit rate: {prefix_hits/len(inputs)*100:.0f}%")
        
        return states

# Define a simple program
def classify_text(s, text):
    s += system("You are a text classifier.")
    s += user(f"Classify: {text}")
    s += select("category", choices=["tech", "sports", "politics", "entertainment"])
    s += user("How confident are you?")
    s += gen("confidence", regex=r"[0-9]|10")

# Run batch
runner = BatchRunner()
inputs = [
    {"text": "New iPhone released today"},
    {"text": "Team wins championship"},
    {"text": "AI breakthrough in research"},
    {"text": "Movie premiere tonight"},
    {"text": "Quantum computing advance"},
]

states = runner.run_batch(classify_text, inputs)

print("\nResults:")
for i, (inp, state) in enumerate(zip(inputs, states)):
    print(f"  {inp['text']:<35s} -> {state['category']}, confidence={state['confidence']}")

## 5. Source Code Architecture

### 5.1 Intermediate Representation (IR)

```python
# sglang/lang/ir.py — Key IR nodes

class SglExpr:
    """Base class for SGLang IR expressions."""
    pass

class SglGen(SglExpr):
    """Generation expression."""
    def __init__(self, name, max_tokens, regex, ...):
        self.name = name
        self.max_tokens = max_tokens
        self.regex = regex

class SglSelect(SglExpr):
    """Selection expression."""
    def __init__(self, name, choices):
        self.name = name
        self.choices = choices

class SglRoleBegin(SglExpr):
    """Begin a role (system/user/assistant)."""
    def __init__(self, role):
        self.role = role

class SglRoleEnd(SglExpr):
    """End a role."""
    pass

class SglFunction:
    """Wraps a Python function as an SGLang program."""
    def __init__(self, func):
        self.func = func
    
    def run(self, **kwargs):
        """Execute the program with given arguments."""
        backend = get_default_backend()
        state = ProgramState(backend)
        self.func(state, **kwargs)
        return state
    
    def run_batch(self, inputs):
        """Execute for multiple inputs in batch."""
        backend = get_default_backend()
        states = []
        for inp in inputs:
            state = ProgramState(backend)
            self.func(state, **inp)
            states.append(state)
        return states
```

### 5.2 Interpreter

```python
# sglang/lang/interpreter.py — Executes IR nodes

class ProgramState:
    def __init__(self, backend):
        self.backend = backend
        self.messages = []
        self.variables = {}
    
    def __iadd__(self, expr):
        if isinstance(expr, str):
            self._append_text(expr)
        elif isinstance(expr, SglGen):
            result = self.backend.generate(
                self.messages, expr.max_tokens, expr.regex
            )
            self.variables[expr.name] = result
            self._append_text(result)
        elif isinstance(expr, SglSelect):
            result = self.backend.select(
                self.messages, expr.choices
            )
            self.variables[expr.name] = result
            self._append_text(result)
        return self
```

In [ ]:
# Trace the IR nodes generated by a program

class IRTracer:
    """Traces IR nodes generated during program execution."""
    
    def __init__(self):
        self.ir_nodes = []
    
    def trace_program(self, program_fn, **kwargs):
        """Execute program and record IR nodes."""
        state = TracingState(self)
        program_fn(state, **kwargs)
        return self.ir_nodes

class TracingState:
    def __init__(self, tracer):
        self.tracer = tracer
    
    def __iadd__(self, other):
        if isinstance(other, str):
            self.tracer.ir_nodes.append({"type": "text", "content": other[:60]})
        elif isinstance(other, SglRole):
            self.tracer.ir_nodes.append({"type": "role", "role": other.role, "content": other.content[:60]})
        elif isinstance(other, SglGenToken):
            self.tracer.ir_nodes.append({"type": "gen", "name": other.name, "params": repr(other)})
        elif isinstance(other, SglSelectToken):
            self.tracer.ir_nodes.append({"type": "select", "name": other.name, "choices": other.choices})
        return self

# Trace a program
def research_assistant(s, topic):
    s += system("You are a research assistant specializing in AI.")
    s += user(f"Research the topic: {topic}")
    s += gen("initial_thoughts", max_tokens=200)
    s += user("What are the key challenges?")
    s += gen("challenges", max_tokens=150)
    s += user("Is this topic well-researched?")
    s += select("maturity", choices=["emerging", "growing", "mature", "declining"])
    s += user("Write a one-paragraph summary.")
    s += gen("summary", max_tokens=100, stop="\n\n")

tracer = IRTracer()
nodes = tracer.trace_program(research_assistant, topic="Quantum Machine Learning")

print("IR Node Trace")
print("=" * 70)
for i, node in enumerate(nodes):
    if node["type"] == "text":
        print(f"  {i+1}. TEXT: {node['content']!r}")
    elif node["type"] == "role":
        print(f"  {i+1}. ROLE({node['role']}): {node['content']!r}")
    elif node["type"] == "gen":
        print(f"  {i+1}. GEN:  {node['params']}")
    elif node["type"] == "select":
        print(f"  {i+1}. SELECT({node['name']}): {node['choices']}")

print(f"\nTotal IR nodes: {len(nodes)}")
print(f"  Text/Role nodes: {sum(1 for n in nodes if n['type'] in ['text', 'role'])}")
print(f"  Gen nodes: {sum(1 for n in nodes if n['type'] == 'gen')}")
print(f"  Select nodes: {sum(1 for n in nodes if n['type'] == 'select')}")

## 6. Backend Connectors

SGLang programs can run on different backends:

```python
# sglang/lang/backend/

# 1. SGLang Runtime (SRT) backend — fastest
runtime = sgl.Runtime(model_path="meta-llama/Llama-3.1-8B-Instruct")
sgl.set_default_backend(runtime)

# 2. OpenAI backend
backend = sgl.OpenAI("gpt-4")
sgl.set_default_backend(backend)

# 3. Anthropic backend
backend = sgl.Anthropic("claude-3-opus")
sgl.set_default_backend(backend)

# 4. Runtime endpoint (connect to running SRT server)
backend = sgl.RuntimeEndpoint("http://localhost:30000")
sgl.set_default_backend(backend)
```

In [ ]:
# Show backend architecture

backends = {
    "sgl.Runtime": {
        "description": "In-process SGLang Runtime (fastest)",
        "usage": 'sgl.Runtime(model_path="meta-llama/Llama-3.1-8B")',
        "features": ["RadixAttention", "Prefix sharing", "Constrained decoding", "CUDA graphs"],
        "latency": "Lowest (no network)",
    },
    "sgl.RuntimeEndpoint": {
        "description": "Connect to running SRT server",
        "usage": 'sgl.RuntimeEndpoint("http://localhost:30000")',
        "features": ["Same as Runtime", "Network overhead", "Multi-client"],
        "latency": "Low (local network)",
    },
    "sgl.OpenAI": {
        "description": "OpenAI API backend",
        "usage": 'sgl.OpenAI("gpt-4o")',
        "features": ["Cloud-hosted", "No GPU needed", "Rate limited"],
        "latency": "Medium (internet)",
    },
    "sgl.Anthropic": {
        "description": "Anthropic API backend",
        "usage": 'sgl.Anthropic("claude-3-opus")',
        "features": ["Cloud-hosted", "No GPU needed", "Rate limited"],
        "latency": "Medium (internet)",
    },
}

print("SGLang Backend Options")
print("=" * 70)

for name, info in backends.items():
    print(f"\n  {name}")
    print(f"    {info['description']}")
    print(f"    Usage: {info['usage']}")
    print(f"    Features: {', '.join(info['features'])}")
    print(f"    Latency: {info['latency']}")

## 7. Summary

### Key Takeaways

1. **SGLang DSL** treats LLM interactions as **programs**, not single prompts
2. **Core primitives**: `gen()`, `select()`, `fork()/join()`, role markers
3. **Named outputs** make it easy to access results from multi-step programs
4. **Batch execution** with RadixAttention automatically shares common prefixes
5. **Multiple backends**: SRT, OpenAI, Anthropic, RuntimeEndpoint
6. **Advanced patterns**: chain-of-thought, tree-of-thought, self-consistency

### Next Chapter

Chapter 5.10 will explore **Multi-Modal Support** — how SGLang handles vision-language models.

## Exercises

### Exercise 1: Build a Debate Program
Write an SGLang program where two LLM "agents" debate a topic, taking turns arguing for and against.

### Exercise 2: Implement fork/join
Extend the `SglState` simulator to support `fork()` and `join()` operations.

### Exercise 3: RAG Pipeline
Write an SGLang program that implements a Retrieval-Augmented Generation pipeline:
1. Generate search queries from the user's question
2. (Simulate) retrieve documents
3. Generate answer using retrieved context

### Exercise 4: Compiler Optimization
Design optimizations for the SGLang compiler that could:
- Detect and batch independent `gen()` calls
- Pre-compute shared prefixes across batch instances
- Optimize away unnecessary role markers

In [ ]:
# Exercise 1 starter

def debate_program(s, topic, num_rounds=3):
    """TODO: Implement a debate between two LLM agents."""
    s += system("You will simulate a debate between two experts.")
    
    for round_num in range(num_rounds):
        # TODO: Agent 1 argues FOR
        s += user(f"Round {round_num + 1}: Argue FOR {topic}")
        s += gen(f"for_{round_num}", max_tokens=150)
        
        # TODO: Agent 2 argues AGAINST
        s += user(f"Round {round_num + 1}: Now argue AGAINST {topic}")
        s += gen(f"against_{round_num}", max_tokens=150)
    
    # TODO: Judge the debate
    s += user("Who won the debate?")
    s += select("winner", choices=["for", "against", "tie"])

# Test
state = SglState()
debate_program(state, "AI will replace most human jobs")
state.display_trace()
print(f"\nWinner: {state['winner']}")